# Sparse vectors in Vespa with BGE-M3

BGE-M3 produces three representations per text. This notebook looks at the **sparse** one (`lexical_weights`):

| representation | shape | captures |
|---|---|---|
| dense | 1024 floats | overall meaning |
| **sparse** | `{token_id: weight}` for the tokens that occur in the text | which vocabulary items matter, and how much |
| multi-vector (ColBERT) | one vector per token | token-level interaction |

A sparse vector looks like a BM25 bag of words, but the weights are **learned** by the model instead of computed from corpus statistics, and the tokens are **sub-words**, so spelling variants such as `autoscaling` / `auto scaling` share pieces.

Prerequisites: Vespa running with the application in `simple_app/` deployed (the schema contains the sparse fields), run from the repository root. Set `VESPA_URL` / `VESPA_PORT` if Vespa is not on `localhost:8080`.

In [1]:
import os, sys, json
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

from vespa.application import Vespa
from FlagEmbedding import BGEM3FlagModel

VESPA_URL = os.environ.get("VESPA_URL", "http://localhost")
VESPA_PORT = int(os.environ.get("VESPA_PORT", "8080"))
app = Vespa(url=VESPA_URL, port=VESPA_PORT)
model = BGEM3FlagModel("BAAI/bge-m3")
tokenizer = model.tokenizer


## 1. What a sparse vector looks like

In [2]:
import pandas as pd

def sparse_vector(text):
    return {k: float(v) for k, v in model.encode([text], return_dense=False, return_sparse=True,
                                                   return_colbert_vecs=False)["lexical_weights"][0].items()}

def show_sparse(text):
    sv = sparse_vector(text)
    rows = [(tokenizer.decode([int(t)]).strip(), t, round(w, 4)) for t, w in sv.items()]
    return pd.DataFrame(rows, columns=["token", "token_id", "weight"]).sort_values("weight", ascending=False).reset_index(drop=True)

show_sparse("Fix error VESP-4022 by making feeding quicker")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,token,token_id,weight
0,feed,61590,0.2285
1,22,4015,0.2167
2,ESP,153726,0.2163
3,quick,63773,0.2104
4,er,56,0.2062
5,error,18499,0.1963
6,making,20662,0.1924
7,V,310,0.1749
8,Fix,96746,0.1692
9,-40,41734,0.1594


Things to notice:

* The key is the **token id** of the XLM-R sentencepiece vocabulary (about 250k entries), so the vector has hundreds of thousands of possible dimensions but only a handful of non-zero ones.
* The rare identifier is split into several sub-word pieces (`VE`, `SP`, `402`, ...). Every piece receives its own weight.
* Filler words (`by`, `making`) get small weights. This is the learned equivalent of IDF.

In [3]:
a = show_sparse("autoscaling of content nodes")
b = show_sparse("auto scaling of nodes")
print("autoscaling of content nodes"); display(a)
print("auto scaling of nodes"); display(b)
print("shared token ids:", sorted(set(a.token_id) & set(b.token_id)))

autoscaling of content nodes


,token,token_id,weight
0,content,10941,0.2466
1,sca,16275,0.2360
2,auto,1809,0.1846
3,no,110,0.1646
4,ling,2069,0.1004
5,des,988,0.0941
6,of,111,0.0097


auto scaling of nodes


,token,token_id,weight
0,auto,1809,0.2213
1,scal,117906,0.2199
2,no,110,0.1833
3,des,988,0.1317
4,ing,214,0.0903
5,of,111,0.0397


shared token ids: ['110', '111', '1809', '988']


The two spellings share sub-word tokens although **no whole word is shared** except `of`. A classic BM25 index (whole-word tokens + stemming) would treat `autoscaling` and `auto scaling` as unrelated.

## 2. The relevance score is a sparse dot product

`score(q, d) = sum over shared tokens of  q_weight * d_weight`. Tokens that appear in only one of the two vectors contribute nothing.

In [4]:
def explain_dot(query, doc):
    q, d = sparse_vector(query), sparse_vector(doc)
    rows = [(tokenizer.decode([int(t)]).strip(), round(q[t], 3), round(d[t], 3), round(q[t] * d[t], 4)) for t in q if t in d]
    df = pd.DataFrame(rows, columns=["token", "q_weight", "d_weight", "product"]).sort_values("product", ascending=False)
    print("score =", round(df["product"].sum(), 4))
    return df

explain_dot("auto scaling of nodes", "Autoscaling adjusts the number of content nodes based on load and data volume.")

score = 0.0622


,token,q_weight,d_weight,product
1,no,0.183,0.196,0.0359
2,des,0.132,0.188,0.0248
0,of,0.040,0.038,0.0015


In [5]:
explain_dot("auto scaling of nodes", "Manual capacity planning estimates the node count needed for a target query rate.")

score = 0.0355


,token,q_weight,d_weight,product
0,no,0.183,0.194,0.0355


## 3. Storing sparse vectors in Vespa

[`doc.sd`](simple_app/schemas/doc.sd) has three extra fields:

```
field sparse_embedding type tensor<float>(t{}) {      # exact weights, used for scoring
    indexing: attribute | summary
}
field title_sparse type tensor<float>(t{}) {          # same for the title only
    indexing: attribute | summary
}
field sparse_tokens type weightedset<string> {        # integer weights, used for retrieval
    indexing: attribute | summary
    attribute: fast-search
}
```

Why two representations of the same vector?

* A **mapped tensor** `tensor<float>(t{})` keeps the float weights and can be multiplied with the query tensor in a rank expression: `sum(query(qs) * attribute(sparse_embedding))`. It cannot be searched efficiently.
* A **weightedset with `fast-search`** builds an inverted index, which the `wand` and `dotProduct` operators use to find candidates without scanning all documents. Weights must be integers, so we multiply by `SPARSE_SCALE = 1000`.

The query side is also sent twice: as a tensor input (`input.query(qs)`) for scoring, and as the weighted token list inside the YQL for retrieval.

In [6]:
from constants import SPARSE_SCALE

SAMPLE_DOCS = {
    "nb-1": "Autoscaling adjusts the number of content nodes based on load and data volume.",
    "nb-2": "Manual capacity planning estimates the node count needed for a target query rate.",
    "nb-3": "Error VESP-4021 means the document exceeds the maximum document size.",
    "nb-4": "Error VESP-4022 means a feed operation timed out.",
}

for doc_id, text in SAMPLE_DOCS.items():
    sv = sparse_vector(text)
    resp = app.feed_data_point(schema="doc", data_id=doc_id, fields={
        "id": doc_id, "title": text, "body": text, "summary_text": text, "semantic_text": text, "category": "demo",
        "embedding": model.encode([text], return_dense=True, return_sparse=False, return_colbert_vecs=False)["dense_vecs"][0].tolist(),
        "title_embedding": [0.0] * 1024, "body_embedding": [0.0] * 1024,
        "sparse_embedding": {"cells": sv},
        "title_sparse": {"cells": sv},
        "sparse_tokens": {t: max(1, int(round(w * SPARSE_SCALE))) for t, w in sv.items()},
    })
    print(doc_id, resp.status_code)

nb-1 200


nb-2 200


nb-3 200


nb-4 200


## 4. Querying: `wand` vs `dotProduct`

Both operators take the weighted query tokens. They find documents that share at least one token and expose the sum of weight products as `rawScore(sparse_tokens)`.

* `wand` (weak AND) uses `targetHits` and skips documents that cannot enter the top-k. Use it for large corpora.
* `dotProduct` is exhaustive over all documents that match any token. Simple and exact, slower on big data.

In [7]:
def sparse_query(text, operator="wand", top_k=None, ranking="sparse_dot", hits=5):
    sv = sparse_vector(text)
    if top_k:
        sv = dict(sorted(sv.items(), key=lambda kv: kv[1], reverse=True)[:top_k])
    weights = ", ".join(f'"{t}": {max(1, int(round(w * SPARSE_SCALE)))}' for t, w in sv.items())
    if operator == "wand":
        predicate = f'([{{"targetHits": {hits}}}]wand(sparse_tokens, {{{weights}}}))'
    else:
        predicate = f"dotProduct(sparse_tokens, {{{weights}}})"
    yql = f"select id, title from doc where {predicate}"
    resp = app.query(yql=yql, ranking=ranking, hits=hits, body={"input.query(qs)": {"cells": sv}})
    rows = [(h["fields"]["id"], round(h["relevance"], 4), h["fields"]["title"][:70],
             {k: round(v, 3) for k, v in h["fields"].get("matchfeatures", {}).items()}) for h in resp.hits]
    return yql, pd.DataFrame(rows, columns=["id", "relevance", "title", "match-features"])

yql, df = sparse_query("auto scaling of nodes")
print(yql[:300], "...")
df

select id, title from doc where ([{"targetHits": 5}]wand(sparse_tokens, {"1809": 221, "117906": 220, "214": 90, "111": 40, "110": 183, "988": 132})) ...


,id,relevance,title,match-features
0,sp-008,0.1151,autoscaling content clusters,{'sparse_score': 0.115}
1,nb-1,0.0622,Autoscaling adjusts the number of content node...,{'sparse_score': 0.062}
2,sp-030,0.0418,release notes 8.314.17,{'sparse_score': 0.042}
3,sp-034,0.0409,release notes 8.340.9,{'sparse_score': 0.041}
4,sp-032,0.0408,release notes 8.327.8,{'sparse_score': 0.041}


In [8]:
_, df = sparse_query("auto scaling of nodes", operator="dotProduct", ranking="sparse_rawscore")
df

,id,relevance,title,match-features
0,sp-008,114849.0,autoscaling content clusters,{'sparse_raw': 114849.0}
1,nb-1,62204.0,Autoscaling adjusts the number of content node...,{'sparse_raw': 62204.0}
2,sp-030,41745.0,release notes 8.314.17,{'sparse_raw': 41745.0}
3,sp-032,40881.0,release notes 8.327.8,{'sparse_raw': 40881.0}
4,sp-034,40800.0,release notes 8.340.9,{'sparse_raw': 40800.0}


Both return the same documents; `rawScore` is the integer-scaled version of the float dot product (`relevance` differs by the factor 1000 squared).

## 5. Pruning the query

Dropping the lowest-weight query tokens makes retrieval cheaper. The cost is recall when the dropped token was the one that mattered.

In [9]:
for k in (None, 4, 2):
    _, df = sparse_query("what shipped in release 8.352.3 of the container", top_k=k, hits=3)
    print("top_k =", k); display(df[["id", "relevance", "title"]])

top_k = None


,id,relevance,title
0,sp-035,0.2556,release notes 8.352.3
1,sp-032,0.1480,release notes 8.327.8
2,sp-031,0.1477,release notes 8.320.4


top_k = 4


,id,relevance,title
0,sp-035,0.2242,release notes 8.352.3
1,sp-034,0.1150,release notes 8.340.9
2,sp-032,0.1145,release notes 8.327.8


top_k = 2


,id,relevance,title
0,sp-035,0.1254,release notes 8.352.3
1,sp-032,0.0689,release notes 8.327.8
2,sp-034,0.0681,release notes 8.340.9


## 6. Next

[`vespa_sparse_strategies.ipynb`](vespa_sparse_strategies.ipynb) turns these building blocks into evaluation strategies (sparse only, sparse + dense, sparse + keyword, all three) and shows each on queries that favour it.

In [10]:
# clean up the demo documents
for doc_id in SAMPLE_DOCS:
    app.delete_data(schema="doc", data_id=doc_id)